# Web-Gold-40K existing-data improvement audit

This CPU-only notebook creates reviewer-ready bbox and weak-class artifacts from the existing attached dataset. It streams a nested ZIP in place, reads only train and validation, does not extract or edit the dataset, does not read test rows, and does not touch `kaggle_gold.ipynb`.

Before running, attach `kiyasmahmud/web-gold-40k` with **Add Input**. A GPU is not needed.

In [ ]:
# 1. Pull the Code branch and install only this repository package.
from pathlib import Path
import subprocess
import sys

REPOSITORY = 'https://github.com/Kiyas-Mahmud/webagent.git'
REPO_ROOT = Path('/kaggle/working/webagent')

if (REPO_ROOT / '.git').is_dir():
    subprocess.run(
        ['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', 'Code'],
        check=True,
    )
else:
    subprocess.run(
        ['git', 'clone', '--branch', 'Code', '--single-branch', REPOSITORY, str(REPO_ROOT)],
        check=True,
    )

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', str(REPO_ROOT)],
    check=True,
)
AUDITOR = REPO_ROOT / 'scripts' / 'audit_gold_existing_data.py'
assert AUDITOR.is_file(), f'Auditor not found: {AUDITOR}'
print('Auditor:', AUDITOR)

In [ ]:
# 2. Locate the attached folder or ZIP. Nothing is downloaded or extracted.
INPUT_ROOT = Path('/kaggle/input')
KNOWN_DATASET_ROOT = Path('/kaggle/input/datasets/kiyasmahmud/web-gold-40k')
assert INPUT_ROOT.is_dir(), 'This notebook must run on Kaggle.'

DATA_SOURCE = KNOWN_DATASET_ROOT if KNOWN_DATASET_ROOT.is_dir() else INPUT_ROOT
print('DATA_SOURCE =', DATA_SOURCE)
for item in sorted(DATA_SOURCE.iterdir()):
    kind = 'directory' if item.is_dir() else f'{item.stat().st_size:,} bytes'
    print(' -', item.name, f'({kind})')

In [ ]:
# 3. Registered review-package settings.
SEED = 42
REVIEW_TASKS_PER_CLASS = 100
MINIMUM_VALIDATION_SUPPORT = 100  # Planning threshold, not a performance claim.
MONTAGE_ROWS_PER_CATEGORY = 12
OUTPUT_DIR = Path('/kaggle/working/gold_existing_data_improvement')
REPORT_PATH = OUTPUT_DIR / 'dataset_improvement_audit.json'

print('Review tasks per class:', REVIEW_TASKS_PER_CLASS)
print('Output:', OUTPUT_DIR)

In [ ]:
# 4. Stream train/validation from the attached source and create the package.
command = [
    sys.executable,
    str(AUDITOR),
    '--data-root', str(DATA_SOURCE),
    '--output-dir', str(OUTPUT_DIR),
    '--review-tasks-per-class', str(REVIEW_TASKS_PER_CLASS),
    '--minimum-val-support', str(MINIMUM_VALIDATION_SUPPORT),
    '--montage-rows-per-category', str(MONTAGE_ROWS_PER_CATEGORY),
    '--seed', str(SEED),
]
print('Running:', ' '.join(command))
result = subprocess.run(command, check=False)
assert result.returncode == 0, f'Auditor crashed with exit code {result.returncode}'
assert REPORT_PATH.is_file(), f'Report was not created: {REPORT_PATH}'

In [ ]:
# 5. Verify the non-destructive contract and show the decisions.
import json
import pandas as pd
from IPython.display import display

report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
assert report['status'] == 'PASS'
assert report['test_rows_read'] == 0
assert report['source_records_mutated'] is False
assert report['source_images_copied_or_extracted'] is False

bbox_table = pd.DataFrame(report['bbox_audit']).T[
    ['records', 'bbox_rows', 'valid_bbox_rows', 'invalid_bbox_rows', 'training_disposition']
]
display(bbox_table)

support = pd.Series(
    report['class_coverage']['target_validation_support'],
    name='validation_support',
).to_frame()
display(support)

print('Missing target classes:', report['class_coverage']['missing_target_classes'])
print('Targeted new collection required if claimed:', report['class_coverage']['targeted_new_collection_required'])
print('BBox queue rows:', report['outputs']['bbox_review_queue_rows'])
print('Weak-class queue rows:', report['outputs']['weak_class_review_queue_rows'])
print('TEST ROWS READ:', report['test_rows_read'])

In [ ]:
# 6. Create one small downloadable ZIP containing only reports, CSVs, and montages.
import shutil

ARCHIVE_BASE = Path('/kaggle/working/gold_existing_data_improvement_review_package')
archive_path = Path(shutil.make_archive(
    str(ARCHIVE_BASE),
    'zip',
    root_dir=OUTPUT_DIR,
))
assert archive_path.is_file()
print('Download:', archive_path)
print('Size:', f'{archive_path.stat().st_size / (1024 * 1024):.2f} MB')

## Interpretation

- Invalid bbox rows remain usable for outcome, failure, action, recovery, confidence, and memory supervision; only bbox loss is masked.
- No bbox is corrected automatically because the exported split schema does not contain scroll-offset/document-coordinate evidence.
- Reviewers fill proposed values only from replay or source evidence and require second-person confirmation.
- Existing nonzero classes are reviewed before deciding to collect more data.
- Zero-support `RETRY` or `ABORT` requires real targeted trajectories only if the thesis trains or claims those strategies.
- Do not inspect or use the locked test split during this improvement cycle.